## Базовый пример запуска Gamac

### Импортируем нужные библиотеки

In [ ]:
import sys
sys.path.append('../')

import warnings
import pandas as pd
warnings.filterwarnings('ignore')

from sklearn.datasets import load_iris
from torchvision.datasets import CIFAR100

from gamac.autoclustering import Gamac

### Получим данные для кластеризации

In [2]:
# Импортируем датафрейм для табличных данных
data = load_iris(as_frame=True)

table = data['data']
table.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2


In [ ]:
# Импортируем данные для CIFAR
# Первая загрузка будет долгая так как он грузит из open-source
cifar100 = CIFAR100('../data/cifar', download=True, train=False)

cifar_txt = [f'a photo of {cifar100.classes[img[1]]}' for img in cifar100][:100]
cifar_img = [img[0] for img in cifar100][:100]
cifar_table = pd.DataFrame(cifar100.targets[:100])

### Инициализируем Gamac
- Класс Gamac имеет следующие аргументы:
    - mab_solver: MabSolvers = MabSolvers.SOFTMAX - тип алгоритма multi-arm bandit
    - hyper_optimiser: HyperOptimisers = HyperOptimisers.OPTUNA - Тип оптимизатора, по умолчанию Optuna
    - target_measures: list[str] | None = None - Целевая мера, по умолчанию не указывается - в этом случае происходит автоматический подбор меры
    - time_limit: int | None = None - Время в секундах поиска оптимальной модели кластеризации
    - iter_limit: int | None = 50 - Кол-во итераций поиска оптимальной модели кластеризации
    - batch_size: int = 32 - размер батча для предобработки текста и изображений
    - model_name: str = "openai/clip-vit-large-patch14" - CLIP-based модель для обработки изображений и текста

In [3]:
# По умолчанию кол-во итераций стоит 50, в реальных кейсах число лучше ставить больше
gamac = Gamac(iter_limit=30)

### Запустим поиск оптимального алгоритма кластеризации для табличных данных
Работа происходит в три этапа
1. Обработка и приведение модальностей в единый вектор
2. Определение меры - можно задать явно в target_measures, либо оставить пустым (в этом случае мера будет подобрана автоматически)
3. Поиск алгоритма кластеризации с наилучшими по выбранным мерам гиперпараметрами
- В ходе перебора логируются успешные и неуспешные запуски перебора
- Неуспешные запуски появляются в ходе получения некорректных гиперпараметров алгоритма под конкретный датасет

### Базовый пример для табличных данных

In [15]:
dataset, best_model = gamac.run(table=table, target_measures=["SYM"])

Detected measures: [<Internal.SYM: ('sym', <function sym at 0x78001a128400>)>]
=== MEASURES 0.0007832050323486328s, {'SYM': 0.0024156754134871316} ===
=== MEASURES 0.0008814334869384766s, {'SYM': 0.5328366590075437} ===
=== ALGO 0.026301145553588867s, SuccessRun, {'name': 'BisectingKMeans', 'n_clusters': 5, 'max_iter': 100, 'init': 'k-means++', 'tol': 0.0001} ===
=== ALGO 0.05903053283691406s, FailedRun, {'bandwidth': 0.009117170936916556, 'max_iter': 194, 'tol': 3.669129458228e-05} ===
=== ALGO 0.0197296142578125s, FailedRun, {'name': 'DBSCAN', 'eps': 0.4337687881161889, 'eps_sq': 0.18815536154378718, 'min_samples': 12} ===
=== ALGO 0.013265371322631836s, FailedRun, {'min_cluster_size': 9, 'min_samples': 6} ===
=== ALGO 0.279895544052124s, FailedRun, {'threshold': 0.385352646235432, 'branching_factor': 79, 'n_clusters': 14} ===
=== MEASURES 0.000514984130859375s, {'SYM': 0.1074089163666564} ===
=== ALGO 0.0021512508392333984s, SuccessRun, {'name': 'KMeans', 'n_clusters': 2, 'max_iter'

### На выходе получаем итоговый датасет после обработки

In [16]:
print(dataset.shape)
print(dataset[:10])

(150, 4)
[[-0.9006812   1.0190043  -1.3402265  -1.3154444 ]
 [-1.1430169  -0.13197948 -1.3402265  -1.3154444 ]
 [-1.3853526   0.32841405 -1.397064   -1.3154444 ]
 [-1.5065205   0.09821729 -1.2833891  -1.3154444 ]
 [-1.021849    1.249201   -1.3402265  -1.3154444 ]
 [-0.53717756  1.9397914  -1.1697142  -1.0521799 ]
 [-1.5065205   0.7888076  -1.3402265  -1.1838121 ]
 [-1.021849    0.7888076  -1.2833891  -1.3154444 ]
 [-1.7488563  -0.36217624 -1.3402265  -1.3154444 ]
 [-1.1430169   0.09821729 -1.2833891  -1.4470764 ]]


### Также получаем информацию о лучшей модели - ее наилучшей конфигурацией и финального значения мер

In [17]:
print(best_model.estimation)

{<Internal.SYM: ('sym', <function sym at 0x78001a128400>)>: 1.0318348783330913}


### Получение меток по лучшему классификатору

In [18]:
print(best_model.model.predict(dataset))

[0 5 5 5 0 2 0 0 5 5 2 0 5 5 2 2 2 0 2 2 0 2 0 0 0 5 0 0 0 5 5 0 2 2 5 5 0
 0 5 0 0 5 5 0 2 5 2 5 2 0 3 3 3 4 3 3 3 4 3 4 4 3 4 3 4 3 3 4 4 4 3 3 3 3
 3 3 3 3 3 4 4 4 4 3 3 3 3 4 3 4 4 3 4 4 4 3 3 3 4 4 1 3 1 3 1 1 4 1 3 6 1
 3 1 4 3 1 3 6 1 4 1 3 1 3 1 1 3 3 1 1 1 6 1 3 3 1 1 3 3 1 1 1 3 1 1 1 3 1
 1 3]


### Пример для мультимодальных данных
Для этого используем данные по CIFAR100

### В данном примере можно использовать все модальности (таблица, текст и изображение)

In [7]:
# В данном примере можно использовать все модальности (таблица, текст и изображение)
dataset, best_model = gamac.run(table=cifar_table, text=cifar_txt, image=cifar_img, target_measures=["SYM"])

print(best_model.model.predict(dataset))

Detected measures: [<Internal.SYM: ('sym', <function sym at 0x7491b5b632e0>)>]
=== MEASURES 0.020325660705566406s, {'SYM': 0.0024465842837614804} ===
=== MEASURES 0.0039052963256835938s, {'SYM': 0.047432096877077994} ===
=== ALGO 0.04848337173461914s, SuccessRun, {'name': 'BisectingKMeans', 'n_clusters': 2, 'max_iter': 100, 'init': 'k-means++', 'tol': 0.0001} ===
=== ALGO 0.044952392578125s, FailedRun, {'bandwidth': 0.10552272829772931, 'max_iter': 282, 'tol': 2.3905190497732368e-05} ===
=== ALGO 0.01232457160949707s, FailedRun, {'name': 'DBSCAN', 'eps': 0.34538837286458346, 'eps_sq': 0.11929312811004453, 'min_samples': 13} ===
=== ALGO 0.04933667182922363s, FailedRun, {'min_cluster_size': 13, 'min_samples': 7} ===
=== ALGO 0.2135779857635498s, FailedRun, {'threshold': 0.44217099090489476, 'branching_factor': 76, 'n_clusters': 9} ===
=== ALGO 0.012706279754638672s, FailedRun, {'name': 'KMeans', 'n_clusters': 14, 'max_iter': 100, 'tol': 0.0001, 'random_state': None} ===
=== MEASURES 0.0

### или использовать только текст + изображение

In [8]:
dataset, best_model = gamac.run(text=cifar_txt, image=cifar_img, target_measures=["SYM"])

print(best_model.model.predict(dataset))

Detected measures: [<Internal.SYM: ('sym', <function sym at 0x7491b5b632e0>)>]
=== MEASURES 0.0068552494049072266s, {'SYM': 0.004652511516292574} ===
=== ALGO 0.10120344161987305s, FailedRun, {'name': 'BisectingKMeans', 'n_clusters': 13, 'max_iter': 100, 'init': 'random', 'tol': 0.0001} ===
=== MEASURES 0.00757598876953125s, {'SYM': 0.0} ===
=== ALGO 0.13168883323669434s, SuccessRun, {'bandwidth': 0.9825855432363375, 'max_iter': 149, 'tol': 5.552544083628199e-05} ===
=== MEASURES 0.007575511932373047s, {'SYM': 0.0} ===
=== ALGO 0.043891191482543945s, SuccessRun, {'name': 'DBSCAN', 'eps': 1.0894532283247824, 'eps_sq': 1.1869083367072906, 'min_samples': 14} ===
=== ALGO 0.014443397521972656s, FailedRun, {'min_cluster_size': 10, 'min_samples': 5} ===
=== MEASURES 0.005692958831787109s, {'SYM': 0.11786515299186327} ===
=== ALGO 0.18747282028198242s, SuccessRun, {'threshold': 0.5661366580612069, 'branching_factor': 57, 'n_clusters': 6} ===
=== ALGO 0.004344940185546875s, FailedRun, {'name':

### Таким образом Gamac может подбирать кластеризацию для:
1. табличных
2. изображений с текстом
3. совместно